# Dataset · Plotly

Run All · Hover, pan, zoom · No upload.

In [ ]:
import json
import sys
import unicodedata
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/plazapd/mappedin.py").is_file()
             and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository or nbs/plazapd.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.plazapd.mappedin import load_tables

DATA_DIR = ROOT / "data/plazapd"
tables = load_tables(DATA_DIR, format="parquet")
manifest = json.loads((DATA_DIR / "manifest.json").read_text())


def text_key(value):
    return "".join(c for c in unicodedata.normalize("NFKD", value).casefold() if c.isalnum())


venue_aliases = {text_key(value) for value in [
    manifest.get("venue_name"), manifest.get("venue_id"), *tables["venues"].name.dropna()
] if isinstance(value, str) and value}


def venue_label(value):
    if isinstance(value, str) and any(alias in text_key(value) for alias in venue_aliases):
        return "Venue"
    return value


for table in tables.values():
    for column in ("name", "description"):
        if column in table:
            table[column] = table[column].map(venue_label).astype(table[column].dtype)

assert tables["entities"].entity_id.is_unique
assert tables["relationships"][["src", "dst"]].isin(set(tables["entities"].entity_id)).all().all()
assert tables["distances"].loc[tables["distances"].status.eq("unreachable"), "distance_m"].isna().all()
pd.options.display.max_colwidth = 50
pd.options.display.max_rows = 25

db = duckdb.connect(":memory:")
for name, frame in tables.items():
    db.register(name, frame)


def sql(query, parameters=None):
    return db.execute(query, parameters).df()

In [ ]:
_ = db.execute("""
CREATE OR REPLACE TEMP VIEW location_spaces AS
SELECT DISTINCT r.src AS location_id, r.dst AS space_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN spaces s ON s.entity_id = r.dst WHERE r.relation = 'spaces';

CREATE OR REPLACE TEMP VIEW space_floors AS
SELECT DISTINCT r.src AS space_id, r.dst AS floor_entity_id
FROM relationships r JOIN spaces s ON s.entity_id = r.src
JOIN floors f ON f.entity_id = r.dst WHERE r.relation = 'floor';

CREATE OR REPLACE TEMP VIEW location_floors AS
SELECT DISTINCT l.location_id, f.floor_entity_id
FROM location_spaces l JOIN space_floors f USING (space_id);

CREATE OR REPLACE TEMP VIEW location_category_links AS
SELECT DISTINCT r.src AS location_id, r.dst AS category_id
FROM relationships r JOIN locations l ON l.entity_id = r.src
JOIN categories c ON c.entity_id = r.dst WHERE r.relation = 'categories';

CREATE OR REPLACE TEMP VIEW node_catalog AS
SELECT e.entity_id, e.id, coalesce(nullif(trim(e.name), ''), e.entity_id) AS label,
       CASE WHEN s.entity_id IS NOT NULL THEN 'store'
            WHEN a.entity_id IS NOT NULL THEN 'amenity'
            WHEN e.collection = 'spaces' THEN 'space'
            WHEN e.collection = 'floors' THEN 'floor'
            WHEN e.collection = 'categories' THEN 'category'
            WHEN e.collection = 'nodes' THEN 'navigation'
            ELSE e.type END AS kind,
       e.hidden, e.navigable, e.amenity
FROM entities e LEFT JOIN stores s USING (entity_id) LEFT JOIN amenities a USING (entity_id);
""")

In [ ]:
nodes = sql("SELECT * FROM node_catalog ORDER BY kind, label, entity_id")
relations = tables["relationships"].copy()
location_spaces = sql("SELECT * FROM location_spaces")
space_floors = sql("SELECT * FROM space_floors")
location_floors = sql("SELECT * FROM location_floors")
location_category_links = sql("SELECT * FROM location_category_links")
floor_names = tables["floors"].set_index("entity_id").name
location_floor_names = location_floors.assign(
    floor=location_floors.floor_entity_id.map(floor_names)
).groupby("location_id").floor.agg(lambda v: " | ".join(sorted(set(v))))
nodes["floors"] = nodes.entity_id.map(location_floor_names).fillna("")
for column in ["hidden", "navigable"]:
    nodes[column] = nodes[column].astype("string").fillna("Unknown")
nodes["amenity"] = nodes.amenity.fillna("")


def membership_graph(store_name="", floor=None, include_amenities=True):
    selected = set(nodes.loc[nodes.kind.eq("store") & nodes.label.str.contains(
        store_name, case=False, regex=False), "entity_id"])
    if include_amenities:
        selected |= set(nodes.loc[nodes.kind.eq("amenity"), "entity_id"])
    selected_floors = set(tables["floors"].entity_id)
    if floor is not None:
        floor_rows = tables["floors"]
        selected_floors = set(floor_rows.loc[
            floor_rows.entity_id.eq(floor) | floor_rows.name.eq(floor), "entity_id"])
        if not selected_floors:
            raise ValueError("Unknown floor; use an exact name or entity ID from the floor table.")
        selected &= set(location_floors.loc[
            location_floors.floor_entity_id.isin(selected_floors), "location_id"])
    selected_spaces = set(location_spaces.loc[location_spaces.location_id.isin(selected), "space_id"])
    if floor is not None:
        selected_spaces &= set(space_floors.loc[
            space_floors.floor_entity_id.isin(selected_floors), "space_id"])
    selected |= selected_spaces | set(space_floors.loc[
        space_floors.space_id.isin(selected_spaces), "floor_entity_id"])
    selected |= set(location_category_links.loc[location_category_links.location_id.isin(selected), "category_id"])
    ns = nodes.loc[nodes.entity_id.isin(selected)].copy().reset_index(drop=True)
    es = relations.loc[relations.src.isin(selected) & relations.dst.isin(selected)].copy().reset_index(drop=True)
    return ns, es


def walking_graph(origin=None, k_stores=5, k_amenities=3):
    if any(not isinstance(k, int) or isinstance(k, bool) or k < 0 for k in (k_stores, k_amenities)):
        raise ValueError("Neighbour counts must be nonnegative integers.")
    candidates = tables["stores"].loc[tables["stores"].navigable.fillna(False)].sort_values(["name", "entity_id"])
    if origin is not None:
        candidates = candidates.loc[candidates.entity_id.eq(origin) | candidates.name.eq(origin)]
    if candidates.empty:
        raise ValueError("Choose a navigable store by exact name or entity ID.")
    source = candidates.iloc[0]
    routes = tables["distances"]
    reachable = routes.loc[routes.from_id.eq(source.id) & routes.status.eq("reachable")].copy()
    selected = []
    for table, count, kind in [("stores", k_stores, "store"), ("amenities", k_amenities, "amenity")]:
        ranked = reachable.loc[reachable.to_id.isin(tables[table].id)].sort_values(["distance_m", "to_id"]).head(count)
        targets = ranked.merge(tables[table][["id", "entity_id"]], left_on="to_id", right_on="id")
        selected.extend({"src": source.entity_id, "dst": row.entity_id,
                         "relation": "nearest_reachable_" + kind, "distance_m": row.distance_m}
                        for row in targets.itertuples(index=False))
    es = pd.DataFrame(selected, columns=["src", "dst", "relation", "distance_m"])
    ids = {source.entity_id} | set(es.dst)
    ns = nodes.loc[nodes.entity_id.isin(ids)].copy().reset_index(drop=True)
    return source.entity_id, ns, es

## Categories × floor

Distinct stores; memberships can overlap. [Plotly heatmaps](https://plotly.com/python/heatmaps/).

In [ ]:
from html import escape
import math
import plotly.express as px
import plotly.graph_objects as go

COLORS = {"category": "#9b59b6", "store": "#2874a6", "amenity": "#148f77", "space": "#d68910", "floor": "#c0392b"}
KIND_ORDER = ["category", "store", "amenity", "space", "floor"]


def network_figure(ns, es, title, origin=None):
    positions = {}
    if origin is None:
        for x, kind in enumerate(KIND_ORDER):
            group = ns.loc[ns.kind.eq(kind)].sort_values(["label", "entity_id"])
            for i, entity_id in enumerate(group.entity_id):
                positions[entity_id] = (x, 0 if len(group)==1 else 1-2*i/(len(group)-1))
    else:
        positions[origin] = (0, 0)
        targets = es.sort_values(["distance_m", "dst"]).dst.tolist()
        for i, entity_id in enumerate(targets):
            angle = 2*math.pi*i/max(len(targets),1)
            positions[entity_id] = (math.cos(angle), math.sin(angle))
    figure = go.Figure()
    labels = ns.set_index("entity_id").label
    x, y, hx, hy, hover = [], [], [], [], []
    for row in es.itertuples(index=False):
        x0, y0 = positions[row.src]
        x1, y1 = positions[row.dst]
        x.extend([x0, x1, None])
        y.extend([y0, y1, None])
        hx.append(0.55*x0+0.45*x1)
        hy.append(0.55*y0+0.45*y1)
        distance = f" · {row.distance_m:.2f} m" if hasattr(row,"distance_m") else ""
        hover.append(f"{escape(labels[row.src])} → {escape(labels[row.dst])}<br>{escape(row.relation)}{distance}")
        if origin is not None:
            figure.add_annotation(x=0.8*x1+0.2*x0, y=0.8*y1+0.2*y0,
                                  ax=0.65*x1+0.35*x0, ay=0.65*y1+0.35*y0,
                                  xref="x",yref="y",axref="x",ayref="y",text="",showarrow=True,arrowhead=2)
    figure.add_trace(go.Scattergl(x=x,y=y,mode="lines",line={"width":0.8,"color":"rgba(120,130,140,0.3)"},hoverinfo="skip",showlegend=False))
    figure.add_trace(go.Scattergl(x=hx,y=hy,mode="markers",marker={"size":5,"color":"#aaa"},text=hover,hovertemplate="%{text}<extra></extra>",showlegend=False))
    for kind in KIND_ORDER:
        group = ns.loc[ns.kind.eq(kind)]
        if group.empty:
            continue
        figure.add_trace(go.Scattergl(
            x=[positions[v][0] for v in group.entity_id],y=[positions[v][1] for v in group.entity_id],
            mode="markers+text" if origin else "markers",text=group.label.map(escape).tolist() if origin else None,
            textposition="top center",name=f"{kind} ({len(group)})",
            customdata=group[["label","entity_id","floors","hidden","navigable","amenity"]].map(escape).to_numpy(),
            hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]}<br>Floors: %{customdata[2]}<br>Hidden: %{customdata[3]}<br>Navigable: %{customdata[4]}<br>%{customdata[5]}<extra></extra>",
            marker={"size":12 if origin else 8,"color":COLORS[kind]},
        ))
    if ns.empty:
        figure.add_annotation(text="No matching locations",showarrow=False)
    figure.update_layout(title=title,template="plotly_white",height=720,hovermode="closest",dragmode="pan",
                         legend={"orientation":"h","y":1.1},margin={"l":25,"r":25,"t":100,"b":40},
                         yaxis={"visible":False},xaxis={"visible":False})
    if origin is None:
        figure.update_xaxes(visible=True,tickmode="array",tickvals=list(range(5)),ticktext=KIND_ORDER,
                            showgrid=False,zeroline=False,range=[-0.3,4.3])
    else:
        figure.update_xaxes(range=[-1.35,1.35])
        figure.update_yaxes(range=[-1.35,1.35],scaleanchor="x",scaleratio=1)
    return figure


def show(figure):
    figure.show(renderer="plotly_mimetype+notebook", config={"scrollZoom": True, "displaylogo": False})

In [ ]:
floor_order = tables["floors"].sort_values(["elevation","entity_id"]).name.tolist()
counts = sql("""
SELECT c.name AS category, f.name AS floor, count(DISTINCT s.entity_id) AS stores
FROM stores s JOIN location_category_links lc ON lc.location_id=s.entity_id
JOIN categories c ON c.entity_id=lc.category_id
JOIN location_floors lf ON lf.location_id=s.entity_id
JOIN floors f ON f.entity_id=lf.floor_entity_id GROUP BY ALL
""")
heatmap_data = counts.pivot(index="category",columns="floor",values="stores").reindex(
    index=sorted(tables["categories"].name),columns=floor_order).fillna(0).astype(int)
heatmap = px.imshow(heatmap_data,text_auto=True,aspect="auto",color_continuous_scale="Blues",
                    labels={"x":"Floor","y":"Category","color":"Distinct stores"},title="Stores by category and floor")
heatmap.update_layout(height=600)
show(heatmap)

## Membership graph

Source relationships; positions group entity types, not coordinates. [Plotly network traces](https://plotly.com/python/network-graphs/).

`STORE_NAME` is a literal substring; `FLOOR=None` includes all floors. Rerun this cell after changing a filter.

In [ ]:
STORE_NAME = ""
FLOOR = "Piso 1"
INCLUDE_AMENITIES = True

membership_nodes, membership_edges = membership_graph(STORE_NAME, FLOOR, INCLUDE_AMENITIES)
display(tables["floors"][["entity_id","name","elevation"]].sort_values(["elevation","entity_id"]))
print(f"{len(membership_nodes):,} nodes · {len(membership_edges):,} directed source edges")
membership_figure = network_figure(membership_nodes,membership_edges,"Membership · hover edge markers for direction")
show(membership_figure)

## Nearest reachable neighbours · metres

Set `ORIGIN` to an exact store name or entity ID; `None` selects the alphabetically first navigable store. Outgoing edges rank measured routes; positions are schematic.

In [ ]:
ORIGIN = None
K_STORES = 5
K_AMENITIES = 3

origin_id, walking_nodes, walking_edges = walking_graph(ORIGIN, K_STORES, K_AMENITIES)
origin_label = nodes.set_index("entity_id").loc[origin_id,"label"]
walking_figure = network_figure(walking_nodes,walking_edges,f"Walking from {origin_label} · nearest reachable neighbours",origin_id)
show(walking_figure)
walking_table = walking_edges.assign(destination=walking_edges.dst.map(nodes.set_index("entity_id").label))
display(walking_table[["destination","relation","distance_m"]].round({"distance_m":2}))